# 01 — Data exploration and audit

**Purpose.** Inspect the raw files of one dataset family (CIC-IDS2017 or UNSW-NB15): header quirks, missing and
infinite values, duplicates, label mapping, class imbalance, constant features and potential shortcut features.

| | |
|---|---|
| **Inputs** | official CSV files in `data/raw/<family>/` (see `data/README.md`) |
| **Outputs** | `reports/<family>/data_audit.json`, figures in `reports/<family>/figures/` |
| **Select dataset** | environment variable `CS_DATASET_FAMILY=cicids2017` (default) or `unsw_nb15` |

Exploration only: nothing in this notebook is used to fit deployable preprocessing. Every number shown is computed
from the files you load — run the notebook to obtain them.

In [ ]:
# --- Project bootstrap: make backend/app/ml and notebooks/nb_utils importable ---------------------
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not ((_root / "backend" / "app" / "ml").is_dir() and (_root / "notebooks").is_dir()):
    if _root.parent == _root:
        raise RuntimeError("Start Jupyter inside the CyberSentinel repository (for example from notebooks/).")
    _root = _root.parent
for _path in (str(_root / "backend"), str(_root / "notebooks")):
    if _path not in sys.path:
        sys.path.insert(0, _path)

from nb_utils import load_config

CFG = load_config()  # all settings come from CS_* environment variables (see nb_utils/config.py)
CFG.describe()

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from app.ml.datasets import get_schema_for_family
from nb_utils import audit, plotting, raw_data
from nb_utils.experiments import save_json

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
SCHEMA = get_schema_for_family(CFG.family)
FEATURES = SCHEMA.feature_names
print(SCHEMA.display_name, "| schema", SCHEMA.schema_version, "|", len(FEATURES), "canonical features")

## 1. Raw files and provenance
SHA-256 checksums of every input file are recorded so that results can be traced to exact inputs.

In [ ]:
files = raw_data.discover_raw_files(CFG.raw_dir, CFG.family)
pd.DataFrame([{"official_split": s, "file": p.name, "size_mb": round(p.stat().st_size / 1e6, 2)} for s, p in files])

## 2. Load through the shared column resolver
Headers are normalized to snake_case by `app.ml.preprocessing.columns` — the same code the API applies to uploads.
CIC-IDS2017's duplicated `Fwd Header Length` column is verified to be identical and dropped; any *other* duplicated
header would be rejected. `CS_MAX_ROWS_PER_FILE` caps rows for quick iterations only (never for final results).

In [ ]:
data, load_report = raw_data.load_family(CFG.raw_dir, CFG.family, max_rows_per_file=CFG.max_rows_per_file)
print("rows:", len(data), "| columns:", data.shape[1])
pd.DataFrame(load_report.files).drop(columns=["unknown_columns"])

In [ ]:
first_file = next(iter(load_report.header_mappings))
changed = [(raw, canon) for raw, canon in load_report.header_mappings[first_file].items() if raw != canon]
print(f"{len(changed)} of {len(load_report.header_mappings[first_file])} headers in {first_file} were normalized")
pd.DataFrame(changed[:15], columns=["raw header", "canonical name"])

## 3. Missing, infinite and invalid values
The shared cleaning step converts non-numeric strings, `±Infinity` and |x| > 1e15 into missing values (counted below).
The median imputer fitted on the *training split* in notebook 02 fills them — identically at inference time.

In [ ]:
cleaning = load_report.cleaning.to_dict()
issues = pd.DataFrame({
    "non_finite": pd.Series(cleaning["non_finite_values"], dtype="float64"),
    "non_numeric": pd.Series(cleaning["non_numeric_values"], dtype="float64"),
    "out_of_range": pd.Series(cleaning["out_of_range_values"], dtype="float64"),
    "empty_in_file": pd.Series(cleaning["missing_values"], dtype="float64"),
}).fillna(0).astype(int)
print("rows audited:", cleaning["rows"])
issues.sort_values(["non_finite", "non_numeric"], ascending=False).head(20)

In [ ]:
audit.missing_table(data, FEATURES).head(20)

## 4. Labels and class imbalance
Raw labels map to canonical fine-grained labels and evaluation categories through a **versioned** mapping. Unknown
labels raise an error instead of being silently re-mapped.

In [ ]:
label_map = (
    data.groupby(["label_original", "label_fine", "label_category"]).size().rename("rows").reset_index()
    .sort_values("rows", ascending=False)
)
print("label mapping version:", SCHEMA.label_mapping_version)
label_map

In [ ]:
distribution = audit.label_distribution(data, "label_category")
print("imbalance ratio (largest / smallest category):", round(audit.imbalance_ratio(data), 1))
print("binary distribution:", data["label_binary"].value_counts().to_dict())
print("rows whose binary label disagrees with attack_cat (UNSW only):",
      sum(f.get("label_inconsistencies", 0) for f in load_report.files))
distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
distribution["count"].sort_values().plot.barh(ax=ax, logx=True)
ax.set_xlabel("rows (log scale)")
ax.set_title(f"{SCHEMA.display_name}: category distribution")
fig.tight_layout()
fig.savefig(CFG.figures_dir / "01_category_distribution.png", dpi=120)

## 5. Duplicates
Exact duplicates inflate scores when they land in both training and test splits; feature vectors with *conflicting* labels are label noise. Notebook 02 removes both before splitting.

In [ ]:
duplicates = audit.duplicate_report(data, FEATURES)
duplicates

## 6. Constant and near-constant features

In [ ]:
constant = audit.constant_columns(data, FEATURES)
top_share = {c: float(data[c].value_counts(normalize=True, dropna=False).iloc[0]) for c in FEATURES}
near_constant = sorted(c for c, share in top_share.items() if share >= 0.999 and c not in constant)
print("constant features:", constant)
print("near-constant features (>= 99.9 % one value):", near_constant)

## 7. Class-conditional medians (numeric features)

In [ ]:
audit.class_conditional_medians(data, SCHEMA.numeric_features).head(25)

## 8. Shortcut-feature screening
Mutual information between each numeric feature and the category on a sample. A very informative *identifier-like*
feature (e.g. CIC-IDS2017 `destination_port`, UNSW-NB15 TTL features) can be a dataset artefact rather than attack
behaviour. Notebook 03 measures its effect with an ablation; `CS_DROP_SHORTCUT_FEATURES=1` removes the candidates.

In [ ]:
mi = audit.shortcut_screen(data, SCHEMA.numeric_features, list(SCHEMA.shortcut_feature_candidates), sample=50_000, seed=CFG.seed)
print("shortcut-feature candidates for this schema:", SCHEMA.shortcut_feature_candidates)
plotting.feature_importance_bar(mi.set_index("feature")["mutual_information"], "Mutual information with category (sample)",
                                CFG.figures_dir / "01_mutual_information.png")
mi.head(20)

## 9. Save the audit report

In [ ]:
audit_report = {
    "family": CFG.family,
    "schema_version": SCHEMA.schema_version,
    "label_mapping_version": SCHEMA.label_mapping_version,
    "files": load_report.files,
    "rows": len(data),
    "row_cap_per_file": CFG.max_rows_per_file,
    "cleaning": cleaning,
    "label_distribution": distribution["count"].to_dict(),
    "binary_distribution": data["label_binary"].value_counts().to_dict(),
    "imbalance_ratio": audit.imbalance_ratio(data),
    "duplicates": duplicates,
    "constant_features": constant,
    "near_constant_features": near_constant,
    "mutual_information_top20": mi.head(20).to_dict(orient="records"),
    "shortcut_candidates": list(SCHEMA.shortcut_feature_candidates),
}
print("saved", save_json(audit_report, CFG.reports_dir / "data_audit.json"))

## Decisions carried into `02_preprocessing`
1. Remove exact duplicates and all copies of feature vectors with conflicting labels **before** splitting.
2. Split before fitting anything; CIC-IDS2017 uses a stratified 70/15/15 split, UNSW-NB15 keeps the publisher's test partition.
3. Remove constant features using the **training split only**.
4. Keep shortcut candidates by default and quantify their influence (notebook 03 ablation), or drop them with `CS_DROP_SHORTCUT_FEATURES=1`.